In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, glob, gc
import numpy as np
import xarray as xr
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# ====== aostools ======
try:
    from aostools_functions import ComputeEPfluxDiv
except ImportError as e:
    raise ImportError("请确保 aostools_functions.py 与本脚本同目录，或已加入 PYTHONPATH") from e

# ============================================================
# Hindcast extracted data root (your SH output)
#   /mnt/soclim0/public_data/weiji/Hindcast/<case>/<var>/<member>.<var>.nc
# ============================================================
HIND_ROOT = "/mnt/soclim0/public_data/weiji/Hindcast"

# ---------------- Settings (match your BWCN logic) ----------------
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

LAT_RANGE = (40.0, 80.0)
DO_UBAR   = False
WAVE      = -1

MAX_WORKERS_CASE = 32   # 每个 case 内并行多少个 member
COMPRESS_LEVEL   = 1    # 与你 ncks -L 1 一致

# ---------------- Helpers ----------------
def sel_latband(da, lat0=40., lat1=80., lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    out = da.sel({lat_name: slc})
    if out.sizes.get(lat_name, 0) == 0:
        raise ValueError("Empty lat selection.")
    return out

def coslat_weighted_mean(da, lat_name="lat"):
    lat = da[lat_name]
    w = np.cos(np.deg2rad(lat))
    return da.weighted(w).mean(lat_name, skipna=True)

def compute_pressure_mid(dsU: xr.Dataset, dsPS: xr.Dataset) -> xr.DataArray:
    """
    p_mid = hyam*P0 + hybm*PS
    注意：你的提取脚本把 PS 单独存到了 PS 文件里，U/V/T 文件通常不含 PS（COORD_VARS 不含 PS）。
    """
    if "PS" not in dsPS:
        raise KeyError("PS not found in PS file dataset.")
    return dsU["hyam"] * dsU["P0"] + dsU["hybm"] * dsPS["PS"]

def interp_profile_logp_4d(v_hyb, p_hyb, p_tgt_pa):
    """
    v_hyb: (time,lev,lat,lon)
    p_hyb: (time,lev,lat,lon)
    out : (time,plev,lat,lon)
    """
    p_tgt_pa = np.asarray(p_tgt_pa, float)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full(p_tgt_pa.shape, np.nan, float)
        p_use, v_use = pcol[m], vcol[m]
        idx = np.argsort(p_use)
        return np.interp(
            np.log(p_tgt_pa),
            np.log(p_use[idx]),
            v_use[idx],
            left=np.nan,
            right=np.nan
        )

    dask_gufunc_kwargs = {"output_sizes": {"plev": len(p_tgt_pa)}}
    out = xr.apply_ufunc(
        _interp_col, v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True, dask="parallelized", output_dtypes=[float],
        dask_gufunc_kwargs=dask_gufunc_kwargs
    )
    return out.assign_coords(plev=("plev", p_tgt_pa))

def _discover_cases():
    cases = sorted([d for d in glob.glob(os.path.join(HIND_ROOT, "[0-9][0-9][0-9][0-9]-[0-9][0-9]*"))
                    if os.path.isdir(d)])
    return [os.path.basename(d) for d in cases]

def _discover_members(case_dir):
    """
    以 U 文件夹为基准：<case>/U/<member>.U.nc
    """
    u_dir = os.path.join(case_dir, "U")
    files = sorted(glob.glob(os.path.join(u_dir, "*.U.nc")))
    members = [os.path.basename(f)[:-5] for f in files]  # strip ".U.nc"
    return members

def _paths_for_member(case, member):
    case_dir = os.path.join(HIND_ROOT, case)
    fU  = os.path.join(case_dir, "U",  f"{member}.U.nc")
    fV  = os.path.join(case_dir, "V",  f"{member}.V.nc")
    fT  = os.path.join(case_dir, "T",  f"{member}.T.nc")
    fPS = os.path.join(case_dir, "PS", f"{member}.PS.nc")
    return fU, fV, fT, fPS

def _out_path(case, member):
    out_dir = os.path.join(HIND_ROOT, case, "EPflux_daily")
    os.makedirs(out_dir, exist_ok=True)
    return os.path.join(out_dir, f"{member}.Fz.nc")

# ---------------- Per-member worker ----------------
def process_one_member(case: str, member: str):
    """
    Hindcast: 没有 OMEGA => ComputeEPfluxDiv(..., w=None)
    输出：Fz(time,plev) 40–80N coslat mean
    """
    out_nc = _out_path(case, member)
    if os.path.exists(out_nc) and os.path.getsize(out_nc) > 0:
        return None  # skip

    fU, fV, fT, fPS = _paths_for_member(case, member)
    for fp in [fU, fV, fT, fPS]:
        if not os.path.exists(fp):
            return f"[{case}] Missing for {member}: {fp}"

    # match your old behavior
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

    try:
        with xr.open_dataset(fU, decode_times=time_coder) as dsU, \
             xr.open_dataset(fV, decode_times=time_coder) as dsV, \
             xr.open_dataset(fT, decode_times=time_coder) as dsT, \
             xr.open_dataset(fPS, decode_times=time_coder) as dsPS:

            if dsU.sizes.get("time", 0) < 2:
                return None

            # 1) pressure mid (needs PS from PS file)
            p_mid = compute_pressure_mid(dsU, dsPS)

            # 2) logp interp to std p
            u_std = interp_profile_logp_4d(dsU["U"], p_mid, PLEV_STD_PA)
            v_std = interp_profile_logp_4d(dsV["V"], p_mid, PLEV_STD_PA)
            t_std = interp_profile_logp_4d(dsT["T"], p_mid, PLEV_STD_PA)

            # 3) to numpy (ComputeEPfluxDiv expects time,p,lat,lon)
            u_np = u_std.transpose("time", "plev", "lat", "lon").values
            v_np = v_std.transpose("time", "plev", "lat", "lon").values
            t_np = t_std.transpose("time", "plev", "lat", "lon").values
            lat_np = dsU["lat"].values

            # 4) EP flux (NO w term for hindcast!)
            # pres in hPa
            _, ep2, _, _ = ComputeEPfluxDiv(
                lat=lat_np,
                pres=(PLEV_STD_PA / 100.0),
                u=u_np, v=v_np, t=t_np,
                w=None,                 # <-- 关键：hindcast 没有 OMEGA
                do_ubar=DO_UBAR, wave=WAVE
            )

            da_ep2 = xr.DataArray(
                ep2,
                coords={"time": dsU["time"], "plev": PLEV_STD_PA, "lat": lat_np},
                dims=("time", "plev", "lat"),
                name="Fz"
            )

            # 5) 40–80N coslat mean => (time, plev)
            da_sub = sel_latband(da_ep2, LAT_RANGE[0], LAT_RANGE[1], "lat")
            Fz_out = coslat_weighted_mean(da_sub).compute()
            Fz_out.name = "Fz"

        ds_out = xr.Dataset({"Fz": Fz_out})
        ds_out.attrs = {
            "description": "Hindcast EP Flux vertical component Fz (daily, 40–80N coslat mean)",
            "lat_range": f"{LAT_RANGE[0]}N-{LAT_RANGE[1]}N",
            "plev_unit": "Pa",
            "Fz_unit": "hPa*m/s^2 (as in aostools ComputeEPfluxDiv)",
            "note": "Hindcast output does NOT include OMEGA; ComputeEPfluxDiv called with w=None (no -<u'w'> term)."
        }

        # NetCDF write (simple + compression)
        encoding = {"Fz": {"zlib": True, "complevel": COMPRESS_LEVEL}}
        ds_out.to_netcdf(out_nc, encoding=encoding)

        # cleanup
        del u_std, v_std, t_std, u_np, v_np, t_np, da_ep2, da_sub, Fz_out, ds_out
        gc.collect()

        return None

    except Exception as e:
        return f"[{case}] Error in {member}: {type(e).__name__}: {e}"

# ---------------- Main driver ----------------
def main():
    cases = _discover_cases()
    if not cases:
        print(f"❌ No cases found under {HIND_ROOT}")
        return

    print(f"Found {len(cases)} cases under {HIND_ROOT}")
    for case in cases:
        case_dir = os.path.join(HIND_ROOT, case)
        members = _discover_members(case_dir)
        if not members:
            print(f"⚠️  [{case}] No members found in U/")
            continue

        print(f"\n====================================================")
        print(f"🚀 Case {case}: {len(members)} members, workers={MAX_WORKERS_CASE}")
        errs = 0

        with ProcessPoolExecutor(max_workers=MAX_WORKERS_CASE) as ex:
            futs = {ex.submit(process_one_member, case, m): m for m in members}
            for f in tqdm(as_completed(futs), total=len(futs), desc=f"EPflux {case}"):
                msg = f.result()
                if msg:
                    errs += 1
                    print("⚠️", msg)

        out_dir = os.path.join(HIND_ROOT, case, "EPflux_daily")
        print(f"✅ Case {case} done. Output: {out_dir} (errors={errs})")

    print("\n🎉 All cases finished.")

if __name__ == "__main__":
    main()
